# GPTQ Qwen + LoRA: dependency groups and physical pruning

Diagnostic experiment on **physical GPU 3 only**. Uses the real `Qwen/Qwen3-30B-A3B-GPTQ-Int4` checkpoint and PEFT LoRA. No optimizer/training, no checkpoint writes.

We distinguish (1) backward through a quantized projection, (2) a TP group rooted in a floating-point adapter, and (3) a structurally valid forward after pruning. Integer storage cannot require gradients; this does not prohibit gradients with respect to floating-point inputs/adapters.

To avoid retaining dequantized weights for all 30B parameters, DG probes are local to the real first-layer Q projection, not the whole model. Success here does **not** establish full attention/model pruning support.

In [1]:
import os
import sys
from pathlib import Path

if "torch" in sys.modules:
    raise RuntimeError("Restart the kernel: CUDA_VISIBLE_DEVICES must be set before importing torch.")
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
os.environ.setdefault("HF_HUB_CACHE", "/glazkov-dev/.cache")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Use the cached checkpoint; do not unexpectedly download tens of GB.
os.environ["HF_HUB_OFFLINE"] = "1"
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lora_transfer_pruning").is_dir())
sys.path.insert(0, str(ROOT))

import copy
import gc
import importlib.metadata
import traceback
import torch
import torch_pruning as tp
from peft import LoraConfig, TaskType, get_peft_model
from gptqmodel import GPTQModel
from gptqmodel.utils.backend import BACKEND

DEVICE = "cuda:0"  # physical cuda:3 after remapping
MODEL = "Qwen/Qwen3-30B-A3B-GPTQ-Int4"
ADAPTER = "probe"
results = {}
torch.manual_seed(42)
for package in ("torch", "transformers", "peft", "gptqmodel", "torch-pruning", "transformer-lens"):
    print(package, importlib.metadata.version(package))
assert torch.cuda.is_available(), "CUDA is unavailable; this experiment has not run."
free, total = torch.cuda.mem_get_info(0)
print("Physical GPU 3:", torch.cuda.get_device_name(0), "free GiB:", free / 2**30)
assert free >= 22 * 2**30, "At least 22 GiB free required on physical GPU 3."


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


KeyboardInterrupt: 

In [2]:
# Explicit training-capable backend: AUTO may select an inference-only kernel.
quantized = GPTQModel.load(
    MODEL, device=DEVICE, backend=BACKEND.GPTQ_TORCH, profile="low_memory",
)
hf_model = quantized.model
hf_model.config.use_cache = False
hf_model.eval()
q_path = "model.layers.0.self_attn.q_proj"
base_q = hf_model.get_submodule(q_path)
print("Base Q:", type(base_q), base_q.in_features, base_q.out_features)
print("qweight:", base_q.qweight.shape, base_q.qweight.dtype, base_q.qweight.requires_grad)
print("Supports training:", getattr(base_q, "SUPPORTS_TRAINING", None))


from_quantized: adapter: None


INFO  Patched transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeSparseMoeBlock -> defuser.modeling.unfused_moe.qwen3_moe.LinearQwen3MoeSparseMoeBlock


INFO  Loader: Auto dtype (native float16): `torch.float16`                     


INFO  QuantizeConfig: Ignoring unknown parameter in the quantization configuration: model_name_or_path.


INFO  QuantizeConfig: Ignoring unknown parameter in the quantization configuration: model_file_base_name.


INFO  QuantizeConfig: offload_to_disk_path auto set to temporary dir `/tmp/gptqmodel_hks20w_3`


... (14 outputs left out)

INFO  Model: Auto-fixed `generation_config` mismatch between model and `generation_config.json`.


INFO  Model: Updated `generation_config`: GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "temperature": 0.6,
  "top_k": 20,
  "top_p": 0.95
}



INFO  Kernel: loaded -> `[TorchLinear]`                                        


Base Q: <class 'gptqmodel.nn_modules.qlinear.torch.TorchLinear'> 2048 4096
qweight: torch.Size([256, 4096]) torch.int32 False
Supports training: True


In [3]:
print(hf_model.get_submodule(q_path))

TorchLinear()


In [4]:
#small hint
# lora_config = LoraConfig(
#     task_type=TaskType.CAUSAL_LM,
#     r=16,
#     lora_alpha=32,          # scaling factor = 2.0 a/r * BA(x)
#     lora_dropout=0.0,
    
#     #target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    
#     #ONLY THAT WAY FOR MOE! IF supply param as module - broken lora adapters
    
#     target_parameters=[ f"model.layers.{layer}.mlp.experts.down_proj"
#         for layer in FRACTION_MLP_LAYERS #layers_to_transform doesnt affect on parameters
#     ], #+ ["gate_up_proj"], #"gate_up_proj"
    
#     layers_to_transform=FRACTION_ATTN_LAYERS,
#     #init_lora_weights="eva", #TODO check other inits?
#     #layer_replication - for duplicate W, but add new BA for new layers
#     bias="none",
#     )

In [5]:
# Only one adapter target is needed for this diagnostic.
config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=[q_path],
    r=16, 
    lora_alpha=16, 
    lora_dropout=0.0, 
    bias="none",
)
peft_model = get_peft_model(hf_model, config, adapter_name=ADAPTER)
q = hf_model.get_submodule(q_path)
print("Wrapped Q:", type(q))
print(q)
assert hasattr(q, "lora_A") and ADAPTER in q.lora_A
# A zero-initialized B would make A/input gradient diagnostics uninformative.
with torch.no_grad():
    q.lora_B[ADAPTER].weight.normal_(std=0.01)
q.eval()
print("Trainable parameters:", [(n, tuple(p.shape)) for n, p in peft_model.named_parameters() if p.requires_grad])
x = torch.randn(1, 4, q.in_features, device=DEVICE, dtype=torch.float16, requires_grad=True)


Wrapped Q: <class 'peft.tuners.lora.gptq.GPTQLoraLinear'>
lora.GPTQLoraLinear(
  (base_layer): TorchLinear()
  (lora_dropout): ModuleDict(
    (probe): Identity()
  )
  (lora_A): ModuleDict(
    (probe): Linear(in_features=2048, out_features=16, bias=False)
  )
  (lora_B): ModuleDict(
    (probe): Linear(in_features=16, out_features=4096, bias=False)
  )
  (lora_embedding_A): ParameterDict()
  (lora_embedding_B): ParameterDict()
  (lora_magnitude_vector): ModuleDict()
  (quant_linear_module): TorchLinear()
)
Trainable parameters: [('base_model.model.model.layers.0.self_attn.q_proj.lora_A.probe.weight', (16, 2048)), ('base_model.model.model.layers.0.self_attn.q_proj.lora_B.probe.weight', (4096, 16))]


In [6]:
from peft.tuners.lora.gptq import GPTQLoraLinear

In [7]:
type(q.base_layer)

gptqmodel.nn_modules.qlinear.torch.TorchLinear

In [8]:
from gptqmodel.nn_modules.qlinear.torch import TorchLinear

In [9]:
q.lora_A.probe

Linear(in_features=2048, out_features=16, bias=False)

In [10]:
type(q.lora_A.probe)

torch.nn.modules.linear.Linear

In [11]:
q.get_base_layer() #analogous q.base_layer

TorchLinear()

In [12]:
# Test base/input gradients separately from LoRA.
def run_check(name, fn):
    try:
        value = fn()
        results[name] = {"status": "ok", **(value or {})}
    except Exception as exc:
        results[name] = {"status": "error", "type": type(exc).__name__, "message": str(exc)}
        traceback.print_exc(limit=5)
    print(name, results[name])
    return results[name]

def base_backward():
    y = q.get_base_layer()(x)
    grad, = torch.autograd.grad(y.float().square().mean(), x)
    assert torch.isfinite(grad).all()
    return {"output_shape": tuple(y.shape), "input_grad_norm": grad.float().norm().item()}

def adapter_backward():
    y = q(x)
    inputs = (x, q.lora_A[ADAPTER].weight, q.lora_B[ADAPTER].weight)
    grads = torch.autograd.grad(y.float().square().mean(), inputs)
    assert all(torch.isfinite(g).all() for g in grads)
    return {"output_shape": tuple(y.shape), "grad_norms_input_A_B": [g.float().norm().item() for g in grads]}

run_check("quantized_base_backward", base_backward)
run_check("lora_backward", adapter_backward)


quantized_base_backward {'status': 'ok', 'output_shape': (1, 4, 4096), 'input_grad_norm': 0.04473235085606575}
lora_backward {'status': 'ok', 'output_shape': (1, 4, 4096), 'grad_norms_input_A_B': [0.04468953236937523, 0.02096228115260601, 0.025731168687343597]}


{'status': 'ok',
 'output_shape': (1, 4, 4096),
 'grad_norms_input_A_B': [0.04468953236937523,
  0.02096228115260601,
  0.025731168687343597]}

In [13]:
y = q(x)
inputs = (x, q.lora_A[ADAPTER].weight, q.lora_B[ADAPTER].weight)
grads = torch.autograd.grad(y.float().square().mean(), inputs)

In [14]:
grads.__len__()

3

In [15]:
grads[0].shape, grads[1].shape, grads[2].shape

(torch.Size([1, 4, 2048]), torch.Size([16, 2048]), torch.Size([4096, 16]))

In [16]:
q.base_layer.weight

_LinearWeightMetadata(device=cuda:0, dtype=torch.float16, shape=(4096, 2048))

In [17]:
q.base_layer.qweight.shape

torch.Size([256, 4096])

In [18]:
q.base_layer.qweight

tensor([[-2021025656,  1954842985, -1972926103,  ...,  1987475862,
          2006608006,  1485273209],
        [-1721399160, -2007734426,  1449170583,  ..., -1717016680,
         -1748329865, -1988257129],
        [-1466534008,  1234992540,  1789294728,  ...,  -979933032,
         -1770555239,  2038781593],
        ...,
        [-1986557800, -1957127307, -1419213207,  ..., -1734043774,
         -1733731963, -2004367196],
        [-2054833799, -1682413160,  2088271514,  ...,  1184274796,
          2035910281,  -896355718],
        [ 1485293956,  1918405469, -1721243918,  ..., -1504422248,
         -1786201913,  1751353428]], device='cuda:0', dtype=torch.int32)

In [19]:
y = q(x)
inputs = (x, q.base_layer.qweight)
grads = torch.autograd.grad(y.float().square().mean(), inputs)

RuntimeError: One of the differentiated Tensors does not require grad

## Existing project helper compatibility

This tests the real GPTQ-LoRA wrapper against the existing LinearBridge selectors. A helper rejection is recorded independently of the raw Torch-Pruning experiment; it is not bypassed and presented as public API support.

In [20]:
from transformer_lens.model_bridge.generalized_components.linear import LinearBridge
from lora_transfer_pruning.adapter.torch_pruning.tp_utils import (
    get_active_module_in_dep_graph_from_linear_bridge, get_device,
)
linear_bridge = LinearBridge(name="q_proj")
linear_bridge.set_original_component(q)

def select_project_target():
    target = get_active_module_in_dep_graph_from_linear_bridge(linear_bridge, tp.prune_linear_out_channels)
    return {"target": type(target).__name__, "is_lora_B": target is q.lora_B[ADAPTER]}

run_check("project_target_selector", select_project_target)
run_check("project_device_selector", lambda: {"device": str(get_device(linear_bridge))})


project_target_selector {'status': 'ok', 'target': 'Linear', 'is_lora_B': True}
project_device_selector {'status': 'ok', 'device': 'cuda:0'}


{'status': 'ok', 'device': 'cuda:0'}

In [21]:
linear_bridge

LinearBridge(2048 -> 4096, bias=False, original_component=GPTQLoraLinear)

## Raw TP groups on independent copies of the real Q projection

Each probe uses a fresh copy; the loaded model stays intact. Rank pruning (`A.out → B.in`) does not change Q output width. Output-channel pruning (`B.out`) does, and therefore needs the quantized base output to change too. We inspect integer buffers, Linear metadata, and the actual post-prune forward. No custom quantized pruner or replacement of GPTQ weights with floating-point weights is used.

In [22]:
q.weight.shape

torch.Size([256, 4096])

In [23]:
def describe(module):
    return {
        "linears": {n: {"in": m.in_features, "out": m.out_features, "weight": tuple(m.weight.shape)}
                    for n, m in module.named_modules() if isinstance(m, torch.nn.Linear)},
        "base_type": type(module.get_base_layer()).__name__,
        "base_in": module.get_base_layer().in_features,
        "base_out": module.get_base_layer().out_features,
        "qweight_shape": tuple(module.get_base_layer().qweight.shape),
    }

def probe_group(kind):
    layer = copy.deepcopy(q)
    probe_input = x.detach().clone().requires_grad_(True)
    base = layer.get_base_layer()
    packed_before = base.qweight.detach().clone()
    graph = tp.DependencyGraph().build_dependency(
        layer,
        example_inputs=(probe_input,),
        # forward_fn=trace_forward,
        # ignored_params=ignored_params,
        #unwrapped_parameters=list(zip(ignored_params, [_] * len(ignored_params)))
    )
    root = {"rank": layer.lora_A[ADAPTER], "output": layer.lora_B[ADAPTER], "base": base}[kind]
    report = {
        "before": describe(layer), "nodes": len(graph.module2node),
        "root_in_DG": root in graph.module2node,
        "base_in_DG": base in graph.module2node,
        "base_registered_pruner": str(graph.get_pruner_of_module(base)),
    }
    results["group_" + kind] = report
    print("DG:", report)
    try:
        group = graph.get_pruning_group(root, tp.prune_linear_out_channels, idxs=[2, 6, 9])
        report["group"] = str(group)
        print(group)
        report["check_pruning_group"] = graph.check_pruning_group(group)
        if not report["check_pruning_group"]:
            report["status"] = "invalid_group"
            return
        group.prune()
        report["prune_returned"] = True
        report["after"] = describe(layer)
        report["packed_weight_unchanged"] = torch.equal(packed_before, base.qweight)
        print("After prune:", report["after"])
        # group.prune returning is not enough: check actual execution and backward.
        out = layer(probe_input)
        grad, = torch.autograd.grad(out.float().square().mean(), probe_input)
        report.update(status="forward_backward_ok", output_shape=tuple(out.shape),
                      input_grad_finite=bool(torch.isfinite(grad).all()))
    except Exception as exc:
        report.update(status="error", error_type=type(exc).__name__, error=str(exc),
                      after=describe(layer), packed_weight_unchanged=torch.equal(packed_before, base.qweight))
        traceback.print_exc(limit=6)
    finally:
        print("RESULT", kind, {k: v for k, v in report.items() if k != "group"})
        del layer, graph, probe_input, packed_before
        gc.collect()
        torch.cuda.empty_cache()

for kind in ("rank", "output"):
    probe_group(kind)


DG: {'before': {'linears': {'lora_A.probe': {'in': 2048, 'out': 16, 'weight': (16, 2048)}, 'lora_B.probe': {'in': 16, 'out': 4096, 'weight': (4096, 16)}}, 'base_type': 'TorchLinear', 'base_in': 2048, 'base_out': 4096, 'qweight_shape': (256, 4096)}, 'nodes': 15, 'root_in_DG': True, 'base_in_DG': False, 'base_registered_pruner': '<torch_pruning.ops.ElementWisePruner object at 0x7531e1aa6a70>'}

--------------------------------
          Pruning Group
--------------------------------
[0] prune_out_channels on lora_A.probe (Linear(in_features=2048, out_features=16, bias=False)) => prune_out_channels on lora_A.probe (Linear(in_features=2048, out_features=16, bias=False)), len(idxs)=3
[1] prune_out_channels on lora_A.probe (Linear(in_features=2048, out_features=16, bias=False)) => prune_out_channels on _Reshape_5(), len(idxs)=3
[2] prune_out_channels on _Reshape_5() => prune_out_channels on _ElementWiseOp_4(MmBackward0), len(idxs)=3
[3] prune_out_channels on _ElementWiseOp_4(MmBackward0) => 

Traceback (most recent call last):
  File "/tmp/ipykernel_405303/3913105221.py", line 46, in probe_group
    out = layer(probe_input)
  File "/glazkov-dev/LoRa-Transfer-Pruning/.venv/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/glazkov-dev/LoRa-Transfer-Pruning/.venv/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
  File "/glazkov-dev/LoRa-Transfer-Pruning/.venv/lib/python3.10/site-packages/peft/tuners/lora/gptq.py", line 96, in forward
    result = result + lora_B(lora_A(dropout(x))) * scaling
RuntimeError: The size of tensor a (4096) must match the size of tensor b (4093) at non-singleton dimension 2


Оно и должно было упасть, так как W не можем прунить как B.

In [24]:
import pandas as pd
display(pd.DataFrame([
    {"probe": name, "status": result.get("status"),
     "prune_returned": result.get("prune_returned", False),
     "packed_weight_unchanged": result.get("packed_weight_unchanged"),
     "error": result.get("error", result.get("message", ""))}
    for name, result in results.items()
]))
print("Peak allocated GiB:", torch.cuda.max_memory_allocated(0) / 2**30)


,probe,status,prune_returned,packed_weight_unchanged,error
0,quantized_base_backward,ok,False,None,
1,lora_backward,ok,False,None,
2,project_target_selector,ok,False,None,
3,project_device_selector,ok,False,None,
4,group_rank,forward_backward_ok,True,True,
5,group_output,error,True,True,The size of tensor a (4096) must match the siz...


Peak allocated GiB: 15.865449905395508


## Interpretation

- A successful backward through GPTQ proves differentiability with respect to floating-point input, **not** trainability of packed integer weights.
- A successful rank group only prunes the adapter's internal rank. PEFT rank metadata/scaling and export consistency require separate handling before using such an adapter in training or saving it.
- For Q output pruning, success requires consistent logical widths of both branches and a working post-prune forward. A group may omit the base, or traverse its operations without providing a quantization-aware structural update.
- GPTQ packing uses bits, groups, scales, zero points and possibly permutations. Correct structural pruning is not generally a matter of dividing channel indices by two.
- Full-model attention pruning additionally needs Q/K/V/O, head dimensions, norms and RoPE to remain consistent. This local experiment does not test those dependencies.

Restart the kernel after inspection to release the loaded model and all diagnostic references.